# ONNM — Kaggle GPU fallback

Runs the **identical** code as a local run. Nothing is reimplemented here: the notebook
clones the repo, installs it, and calls the same scripts, so a result produced on Kaggle is
comparable to one produced locally.

Use this when the local ROCm stack is unavailable — wrong AMD driver, wrong Python version,
or no GPU at all.

**Free quota:** ~30 GPU-hours/week, 12-hour sessions, T4 ×2 or P100.

## Setup (once)

1. Notebook settings → **Accelerator: GPU T4 ×2**
2. Notebook settings → **Internet: On**
3. Upload `BTXRD.zip` as a private Kaggle Dataset named `btxrd`, so the 840 MB download
   happens once rather than every session.

The `kaggle` profile in `configs/base.yaml` already points `data_root` at
`/kaggle/input/btxrd/BTXRD` and switches the loader to `num_workers=2, cache_rate=0.0`
(Linux forks properly but Kaggle has less RAM than the local box).

In [ ]:
# Kaggle ships a working CUDA torch. Do NOT reinstall it -- doing so routinely
# breaks the preinstalled CUDA libraries. Install only the project layer.
!pip install -q monai==1.5.2 pydicom openpyxl

import torch

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
import os

REPO = "https://github.com/<your-user>/OsteoNeuralNetwork-Model.git"  # <-- edit
WORK = "/kaggle/working/OsteoNeuralNetwork-Model"

if not os.path.exists(WORK):
    !git clone -q {REPO} {WORK}
%cd {WORK}
!pip install -q -e .

In [ ]:
# Gates 1 and 2: environment, then data.
!python scripts/verify_env.py
!python scripts/verify_data.py --config configs/base.yaml

## Splits — reproducibility matters here

To compare a Kaggle result against a local one, **copy `data/interim/splits.json` from the
local run** rather than regenerating it. `make_splits.py` prints a `content_hash`; if the two
hashes differ, the runs used different data and their scores are not comparable.

Regenerating with the same seed should reproduce the same split, but copying the file removes
any doubt.

In [ ]:
!python scripts/make_splits.py
!python scripts/overfit_check.py --samples 32 --steps 200   # Gate 6

In [ ]:
!python scripts/train.py \
    --profile kaggle \
    --override configs/densenet121_3class.yaml \
    --tag kaggle

In [ ]:
import glob

ckpt = sorted(glob.glob("reports/kaggle-*/best.pt"))[-1]
print("checkpoint:", ckpt)

!python scripts/evaluate.py --checkpoint {ckpt} --profile kaggle --split test
!python scripts/gradcam_report.py --checkpoint {ckpt} --split test

## Getting results out

Kaggle wipes `/kaggle/working` when the session ends. Commit the notebook (Save Version → Save
& Run All) to persist outputs, or download the checkpoint and metric JSON before the session
closes.

**Do not upload Grad-CAM overlays anywhere public.** They are derived images under BTXRD's
CC BY-NC-ND licence.

In [ ]:
import shutil

run_dir = ckpt.rsplit("/", 1)[0]
shutil.make_archive("/kaggle/working/onnm_run", "zip", run_dir)
print("Download /kaggle/working/onnm_run.zip from the Output tab.")